# Email Spam Detection
**Task 4 - OASIS INFOBYTE Data Science Internship**

Classify emails as spam or ham (non-spam) using Natural Language Processing and Machine Learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_style('whitegrid')
%matplotlib inline

---
## 1. Load the Dataset

In [ ]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df.head()

In [ ]:
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df.head()

In [ ]:
df.info()

In [ ]:
df['label'].value_counts()

---
## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='label', palette='Set2')
plt.title('Spam vs Ham Distribution')
plt.show()

In [ ]:
df['length'] = df['message'].apply(len)

plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='length', hue='label', bins=50, kde=True)
plt.title('Message Length Distribution')
plt.show()

In [ ]:
df.groupby('label')['length'].describe()

---
## 3. Data Preprocessing

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

df['clean_message'] = df['message'].apply(clean_text)
df.head()

In [ ]:
X = df['clean_message']
y = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples: {X_test.shape[0]}')

---
## 4. Model Training & Evaluation

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=200),
    'SVM': SVC(kernel='linear')
}

results = {}
for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'\n{"="*40}')
    print(f'{name}')
    print(f'Accuracy: {acc:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred))
    print(f'Confusion Matrix:')
    print(confusion_matrix(y_test, y_pred))

In [ ]:
plt.figure(figsize=(8, 5))
colors = ['#2E86AB', '#A23B72', '#F18F01']
bars = plt.bar(results.keys(), results.values(), color=colors)
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.title('Model Comparison - Email Spam Detection')
for bar, acc in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', fontweight='bold')
plt.show()

---
## 5. Test with Custom Messages

In [ ]:
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
print(f'Best model: {best_model_name} ({results[best_model_name]:.2%} accuracy)')

test_messages = [
    "Congratulations! You've won a free iPhone. Click here to claim now.",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account has been compromised. Update your password immediately."
]

for msg in test_messages:
    cleaned = clean_text(msg)
    vec = vectorizer.transform([cleaned])
    pred = best_model.predict(vec)[0]
    print(f"{'SPAM' if pred == 1 else 'HAM':>5} -> {msg}")